# Ogefmeeting — Labo IA v1 (Deepgram STT live)

**Objectif :** parler dans le micro → voir le **texte en temps réel** pendant la réunion.

**Langue principale :** français (`fr`)

| Étape | Outil | Mode |
|-------|-------|------|
| Ce notebook | **Deepgram** WebSocket | **Streaming temps réel** |
| Prod (app) | Deepgram via backend Node | Idem + sauvegarde Supabase |
| CR (plus tard) | OpenAI GPT | À la fin de la réunion |

> **Sécurité :** ne mettez jamais votre clé API dans le notebook. Utilisez uniquement `IA/.env`.

## 0. Configuration

1. `pip install -r requirements.txt`
2. Copier `.env.example` → `.env`
3. Coller votre clé Deepgram : `DEEPGRAM_API_KEY=...`
4. Vérifier que le micro est activé dans Windows
5. Redémarrer le noyau Jupyter après modification du `.env`

In [1]:
import json
import os
import queue
import threading
import time
from pathlib import Path
from urllib.parse import urlencode

import nest_asyncio
import sounddevice as sd
import websockets
from dotenv import load_dotenv

nest_asyncio.apply()
load_dotenv()

IA_DIR = Path.cwd()
SAMPLES = IA_DIR / "samples"
SAMPLES.mkdir(exist_ok=True)

DEEPGRAM_API_KEY = os.getenv("DEEPGRAM_API_KEY")
DEEPGRAM_LANGUAGE = os.getenv("DEEPGRAM_LANGUAGE", "fr")
DEEPGRAM_MODEL = os.getenv("DEEPGRAM_MODEL", "nova-3")
SAMPLE_RATE = 16000

if not DEEPGRAM_API_KEY:
    raise ValueError("Définissez DEEPGRAM_API_KEY dans IA/.env")

print("Modèle Deepgram:", DEEPGRAM_MODEL)
print("Langue STT:", DEEPGRAM_LANGUAGE)
print("Micro par défaut:", sd.query_devices(kind="input"))

Modèle Deepgram: nova-3
Langue STT: fr
Micro par défaut: {'name': 'Microphone (Realtek(R) Audio)', 'index': 1, 'hostapi': 0, 'max_input_channels': 4, 'max_output_channels': 0, 'default_low_input_latency': 0.09, 'default_low_output_latency': 0.09, 'default_high_input_latency': 0.18, 'default_high_output_latency': 0.18, 'default_samplerate': 44100.0}


---
## 1. Transcription live (micro → Deepgram)

Même principe que le script Deepgram fourni :
- WebSocket `wss://api.deepgram.com/v1/listen`
- Audio PCM 16 kHz mono (`linear16`)
- Résultats **intermédiaires** (`[Interim]`) puis **finaux** (`[FINAL]`)

**Utilisation :**
1. Exécutez la cellule ci-dessous
2. Parlez en **français**
3. Appuyez sur **Entrée** dans la zone de saisie pour arrêter

In [2]:
import asyncio


def build_deepgram_ws_url() -> str:
    params = {
        "model": DEEPGRAM_MODEL,
        "language": DEEPGRAM_LANGUAGE,
        "encoding": "linear16",
        "sample_rate": str(SAMPLE_RATE),
        "channels": "1",
        "interim_results": "true",
        "punctuate": "true",
        "smart_format": "true",
        "endpointing": "10",
        # diarize=true possible mais pas indispensable pour le test v1
    }
    return f"wss://api.deepgram.com/v1/listen?{urlencode(params)}"


async def transcrire_live_micro() -> dict:
    """Capture micro + streaming Deepgram (temps réel)."""
    audio_queue: queue.Queue[bytes | None] = queue.Queue()
    stop_event = threading.Event()
    segments_finaux: list[str] = []
    erreurs: list[str] = []

    def capture_micro() -> None:
        def callback(indata, _frames, _time, status):
            if status:
                print(status)
            if not stop_event.is_set():
                audio_queue.put(bytes(indata))

        print("🔴 Parlez maintenant. Entrée pour ARRÊTER.")
        with sd.InputStream(
            samplerate=SAMPLE_RATE,
            channels=1,
            dtype="int16",
            blocksize=4096,
            callback=callback,
        ):
            input()
            stop_event.set()
            audio_queue.put(None)

    capture_thread = threading.Thread(target=capture_micro, daemon=True)
    capture_thread.start()

    ws_url = build_deepgram_ws_url()
    headers = {"Authorization": f"Token {DEEPGRAM_API_KEY}"}

    async with websockets.connect(ws_url, additional_headers=headers) as ws:

        async def envoyer_audio() -> None:
            while True:
                chunk = await asyncio.to_thread(audio_queue.get)
                if chunk is None:
                    break
                await ws.send(chunk)
            await ws.send(json.dumps({"type": "CloseStream"}))

        async def recevoir_transcripts() -> None:
            try:
                async for message in ws:
                    payload = json.loads(message)
                    alt = payload.get("channel", {}).get("alternatives", [{}])[0]
                    transcript = (alt.get("transcript") or "").strip()
                    if not transcript:
                        continue

                    is_final = payload.get("is_final", False)
                    prefix = "[FINAL]" if is_final else "[Interim]"
                    print(f"{prefix} {transcript}")

                    if is_final:
                        segments_finaux.append(transcript)
            except Exception as exc:
                erreurs.append(str(exc))

        await asyncio.gather(envoyer_audio(), recevoir_transcripts())

    capture_thread.join(timeout=2)

    return {
        "texte_complet": " ".join(segments_finaux).strip(),
        "segments_finaux": segments_finaux,
        "langue": DEEPGRAM_LANGUAGE,
        "fournisseur": "deepgram",
        "erreurs": erreurs,
    }


print("Appuyez sur Entrée pour DÉMARRER la transcription live…")
input()

resultat_stt = asyncio.run(transcrire_live_micro())

print("\n--- Texte final ---\n")
print(resultat_stt["texte_complet"] or "(vide — parlez plus fort / plus longtemps)")
print(f"\n{len(resultat_stt['segments_finaux'])} segment(s) final(aux)")
if resultat_stt["erreurs"]:
    print("Erreurs:", resultat_stt["erreurs"])

Appuyez sur Entrée pour DÉMARRER la transcription live…


🔴 Parlez maintenant. Entrée pour ARRÊTER.


RuntimeError: Timeout should be used inside a task

---
## 2. (Optionnel) Test sur un flux audio distant

Reproduit le script bash Deepgram (ffmpeg + websocat) mais en Python.
Utile pour vérifier la clé API **sans micro**.

In [ ]:
# Décommentez et exécutez si besoin (nécessite ffmpeg installé sur la machine)
# !ffmpeg -version

# Exemple : flux radio anglais (changez language=en si vous testez ce flux)
# STREAM_URL = "https://playerservices.streamtheworld.com/api/livestream-redirect/CSPANRADIOAAC.aac"

print("Cellule optionnelle — laissez vide pour l'instant. Le test principal est le micro live.")

---
## 3. Prochaine étape (CR OpenAI)

Quand le STT live te convient :
1. On ajoute une cellule **synthèse CR** avec `OPENAI_API_KEY`
2. Puis intégration backend WebSocket + panneau live dans `ReunionLivePage`

### Architecture prod visée

```
[Frontend React — ReunionLivePage]
   MediaRecorder + panneau « Transcription live »
        │ WebSocket (chunks PCM)
        ▼
[Backend Node — transcription-deepgram.service.ts]
   Relais vers wss://api.deepgram.com/v1/listen
        │ INSERT segments
        ▼
[Supabase] transcriptions + segments_transcription

Fin réunion → OpenAI GPT → brouillon CR
```